In [1]:
#PART D : Implement 1
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

data_path = "c4-train.00000-of-01024-30K.json"

corpus = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        
        corpus.append(item.get("text", ""))

print(f"Tổng số lượng documents (N): {len(corpus)}")

Tổng số lượng documents (N): 30000


Kiểm tra Kích thước Matrix & Tính Sparsity (S)

In [2]:
# Tạo TfidfVectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

# Kích thước Matrix (N x V)
N, V = X.shape
print(f"Number of documents (N): {N}")
print(f"Vocabulary size (V): {V}")
print(f"Matrix shape: {X.shape}")

# Tính Sparsity 
nnz = X.nnz  
total_entries = N * V
sparsity = 1.0 - (nnz / total_entries)

print(f"Number of non-zero elements (nnz): {nnz}")
print(f"Matrix Sparsity (S): {sparsity * 100:.4f}%")

Number of documents (N): 30000
Vocabulary size (V): 193540
Matrix shape: (30000, 193540)
Number of non-zero elements (nnz): 4985822
Matrix Sparsity (S): 99.9141%


Kiểm tra inspect vocabulary

In [3]:
feature_names = np.array(vectorizer.get_feature_names_out())


count_vec = CountVectorizer()
X_counts = count_vec.fit_transform(corpus)
df_per_term = (X_counts > 0).sum(axis=0).A1
top_20_df_idx = df_per_term.argsort()[-20:][::-1]
print("Top 20 terms phổ biến nhất (DF):")
print(feature_names[top_20_df_idx])


idf_scores = vectorizer.idf_
top_20_idf_idx = idf_scores.argsort()[-20:][::-1]
print("\nTop 20 terms có IDF cao nhất:")
print(feature_names[top_20_idf_idx])

doc_0_tfidf = X[0].toarray().flatten()
top_20_doc0_idx = doc_0_tfidf.argsort()[-20:][::-1]
print("\nTop 20 terms TF-IDF cao nhất ở Document 0:")
print(feature_names[top_20_doc0_idx])

Top 20 terms phổ biến nhất (DF):
['the' 'and' 'to' 'of' 'in' 'for' 'is' 'with' 'on' 'that' 'this' 'are'
 'it' 'as' 'at' 'from' 'be' 'you' 'by' 'have']

Top 20 terms có IDF cao nhất:
['00000' '00003' '000040' '00005' '0000856166' '0001042' '000116' '00012'
 '00015' '00016' '000165101' '0002' '00022' '000226' '000281' '확인하게'
 '환원되지' '활동' '활동에' '활동을']

Top 20 terms TF-IDF cao nhất ở Document 0:
['bbq' 'class' 'meat' 'balay' 'kcbs' 'lonestar' 'will' 'missoula' 'apron'
 'smoker' 'you' 'timelines' 'trimming' 'spectators' 'cost' 'rangers'
 '22nd' 'beginner' 'beginners' 'culinary']


PART F: Preprocessing Ablation (Thử nghiệm Tiền xử lý)

In [4]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


def pipeline_a_clean(text):
    # Pipeline A (Minimal)
    return text.lower()

def pipeline_b_clean(text):
    # Pipeline B (Normalized)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) 
    return text


from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


vec_a = TfidfVectorizer(lowercase=True)
vec_b = TfidfVectorizer(lowercase=True, stop_words='english', token_pattern=r'(?u)\b\w+\b')
vec_c = TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2)) # Extended: Thêm Bigram

pipelines = {
    "Pipeline A (Minimal)": vec_a,
    "Pipeline B (Normalized)": vec_b,
    "Pipeline C (Extended)": vec_c
}

results_f = []

for name, vec in pipelines.items():
    X_p = vec.fit_transform(corpus)
    N_p, V_p = X_p.shape
    nnz_p = X_p.nnz
    sparsity_p = 1.0 - (nnz_p / (N_p * V_p))
    avg_tokens = nnz_p / N_p # Số lượng từ trung bình có trong 1 doc
    
    results_f.append({
        "Pipeline": name,
        "Vocabulary size": V_p,
        "Average tokens/document": f"{avg_tokens:.2f}",
        "Matrix sparsity": f"{sparsity_p * 100:.4f}%"
    })

df_f = pd.DataFrame(results_f)
print(df_f.to_string(index=False))

               Pipeline  Vocabulary size Average tokens/document Matrix sparsity
   Pipeline A (Minimal)           193540                  166.19        99.9141%
Pipeline B (Normalized)           193521                  125.05        99.9354%
  Pipeline C (Extended)          3976923                  295.01        99.9926%


In [5]:
# Đo đạc đầy đủ các chỉ số cho Bảng 9.4
results_9_4 = []


test_queries = ["medical image classification", "transformer language model", "deep learning healthcare"]

for name, vec in pipelines.items():
    X_p = vec.fit_transform(corpus)
    N_p, V_p = X_p.shape
    nnz_p = X_p.nnz
    sparsity_p = 1.0 - (nnz_p / (N_p * V_p))
    avg_tokens = nnz_p / N_p
    
    
    feature_names = set(vec.get_feature_names_out())
    total_q_words = 0
    oov_q_words = 0
    for q in test_queries:
        words = q.lower().split()
        total_q_words += len(words)
        oov_q_words += sum(1 for w in words if w not in feature_names)
    oov_rate = (oov_q_words / total_q_words) * 100 if total_q_words > 0 else 0
    
    results_9_4.append({
        "Metric": name,
        "Vocabulary size": V_p,
        "Average tokens/document": f"{avg_tokens:.2f}",
        "Matrix sparsity": f"{sparsity_p * 100:.4f}%",
        "OOV rate": f"{oov_rate:.2f}%"
    })

df_9_4 = pd.DataFrame(results_9_4)
print(df_9_4.to_string(index=False))

df_9_4.to_csv("results.csv", index=False)

                 Metric  Vocabulary size Average tokens/document Matrix sparsity OOV rate
   Pipeline A (Minimal)           193540                  166.19        99.9141%    0.00%
Pipeline B (Normalized)           193521                  125.05        99.9354%    0.00%
  Pipeline C (Extended)          3976923                  295.01        99.9926%    0.00%


PART G: Application - Document Search Engine

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer_search = vec_b
X_search = vectorizer_search.fit_transform(corpus)

def search(query, top_k=5):
    
    query_vec = vectorizer_search.transform([query])
    
    sim_scores = cosine_similarity(query_vec, X_search).flatten()
    
    top_indices = sim_scores.argsort()[-top_k:][::-1]
    
    print(f"\n🔍 QUERY: '{query}'")
    print("-" * 75)
    print(f"{'Rank':<5} | {'Doc ID':<8} | {'Similarity':<10} | {'Document Preview'}")
    print("-" * 75)
    
    for rank, idx in enumerate(top_indices, 1):
        preview = corpus[idx][:60].replace('\n', ' ') + "..."
        print(f"{rank:<5} | {idx:<8} | {sim_scores[idx]:<10.4f} | {preview}")

sample_queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare"
]

for q in sample_queries:
    search(q, top_k=5)


🔍 QUERY: 'medical image classification'
---------------------------------------------------------------------------
Rank  | Doc ID   | Similarity | Document Preview
---------------------------------------------------------------------------
1     | 18971    | 0.4237     | The new RTS Environmental Classification system (RTS GLT) is...
2     | 8527     | 0.3537     | History of maize classification. How races used in classific...
3     | 19908    | 0.2609     | Download League Of Legends Wallpapers in high-quality for yo...
4     | 12658    | 0.2490     | This guidance is for pharmacists who handle, use and sell/su...
5     | 15682    | 0.2486     | - Group Image: Provided functionality of group's image, user...

🔍 QUERY: 'transformer language model'
---------------------------------------------------------------------------
Rank  | Doc ID   | Similarity | Document Preview
---------------------------------------------------------------------------
1     | 27936    | 0.4792     | hi, I 

PART H: Evaluation (Đánh giá định lượng)

In [7]:

eval_set = [
    {
        "query": "medical image classification",
        "relevant_ids": [idx for idx, doc in enumerate(corpus) if "medical" in doc.lower() and "image" in doc.lower()][:10]
    },
    {
        "query": "transformer language model",
        "relevant_ids": [idx for idx, doc in enumerate(corpus) if "transformer" in doc.lower() and "model" in doc.lower()][:10]
    },
    {
        "query": "deep learning healthcare",
        "relevant_ids": [idx for idx, doc in enumerate(corpus) if "deep learning" in doc.lower() and "healthcare" in doc.lower()][:10]
    }
]

def evaluate_retrieval(eval_set, top_k=5):
    p_at_k_list = []
    r_at_k_list = []
    mrr_list = []
    
    for item in eval_set:
        query = item["query"]
        relevant_ids = set(item["relevant_ids"])
        
        if not relevant_ids:
            continue
       
        query_vec = vectorizer_search.transform([query])
        sim_scores = cosine_similarity(query_vec, X_search).flatten()
        retrieved_ids = sim_scores.argsort()[-top_k:][::-1]
        
        #  Precision@K
        relevant_retrieved = [doc_id for doc_id in retrieved_ids if doc_id in relevant_ids]
        p_at_k = len(relevant_retrieved) / top_k
        p_at_k_list.append(p_at_k)
        
        #  Recall@K
        r_at_k = len(relevant_retrieved) / len(relevant_ids)
        r_at_k_list.append(r_at_k)
        
        #  MRR (Mean Reciprocal Rank)
        reciprocal_rank = 0.0
        for rank, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in relevant_ids:
                reciprocal_rank = 1.0 / rank
                break
        mrr_list.append(reciprocal_rank)
        
    print(f"Mean Precision@{top_k}: {np.mean(p_at_k_list):.4f}")
    print(f"Mean Recall@{top_k}:    {np.mean(r_at_k_list):.4f}")
    print(f"MRR (Mean Reciprocal Rank): {np.mean(mrr_list):.4f}")

evaluate_retrieval(eval_set, top_k=5)

Mean Precision@5: 0.0667
Mean Recall@5:    0.0556
MRR (Mean Reciprocal Rank): 0.3333


Part I

## AI Contribution Declaration

According to the AI Usage Policy (Part 14), I declare the AI contributions for this lab as follows:

* **AI Assistance:**
  - **Code Generation & Optimization:** AI assisted in drafting initial code templates for text preprocessing pipelines (Pipeline A, B, C), TF-IDF vectorization, sparse matrix metric calculations, and evaluation metrics (Precision@5, MRR) in `experiments.ipynb`.
  - **Debugging & Error Analysis:** AI helped structure the analysis framework for failure cases, identifying lexical overlap issues (e.g., synonym disconnect between "heart attack" and "myocardial infarction").
  - **Formatting & Structuring:** AI provided Markdown layout suggestions for tables and sections in `reflection.md` and `prediction.md`.

* **Student Verification & Modifications:**
  - I independently executed all code cells on the C4 30K dataset and verified all output metrics (Vocabulary size, Average tokens, Matrix sparsity, Similarity scores).
  - I validated the mathematical steps in Part B and cross-checked the experimental results against my initial predictions in Part C.
  - I manually authored the contextual analysis, failure case evaluations, and learning takeaways in this reflection document.

In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Định nghĩa 4 Queries để kiểm thử (2 Good cases, 2 Failure/Bad cases)
analysis_queries = [
    {"type": "Good (Hiệu quả)", "query": "medical image classification"},
    {"type": "Good (Hiệu quả)", "query": "transformer language model"},
    {"type": "Failure (Kém - Đồng nghĩa)", "query": "heart attack treatment"},
    {"type": "Failure (Kém - Khớp rời rạc)", "query": "quick brown fox"}
]

# Sử dụng Pipeline B vectorizer đã fit ở Part F/G
feature_names = np.array(vectorizer_search.get_feature_names_out())

print("=" * 80)
print("PART I: ERROR ANALYSIS - CHẠY THỬ VÀ PHÂN TÍCH 4 QUERIES")
print("=" * 80)

for item in analysis_queries:
    q_type = item["type"]
    query_text = item["query"]
    
    # Transform query
    q_vec = vectorizer_search.transform([query_text])
    sim_scores = cosine_similarity(q_vec, X_search).flatten()
    
    # Lấy Top 5 kết quả
    top_5_idx = sim_scores.argsort()[-5:][::-1]
    
    print(f"\n📌 TYPE: [{q_type}] | QUERY: '{query_text}'")
    print("-" * 80)
    print(f"{'Rank':<5} | {'Doc ID':<8} | {'Similarity':<10} | {'Matched Terms (Lexical Overlap)'}")
    print("-" * 80)
    
    # Lấy các từ trong query có trong vocab
    q_words = [w for w in query_text.lower().split() if w in vectorizer_search.vocabulary_]
    
    for rank, doc_id in enumerate(top_5_idx, 1):
        doc_text = corpus[doc_id].lower()
        # Tìm những từ trong query thực sự xuất hiện trong document này
        matched_terms = [w for w in q_words if w in doc_text]
        
        # Xem đóng góp IDF của các từ khớp
        matched_info = []
        for w in matched_terms:
            idf_val = vectorizer_search.idf_[vectorizer_search.vocabulary_[w]]
            matched_info.append(f"{w}(IDF:{idf_val:.2f})")
            
        matched_str = ", ".join(matched_info) if matched_info else "Không có (Score ~ 0)"
        
        print(f"{rank:<5} | {doc_id:<8} | {sim_scores[doc_id]:<10.4f} | {matched_str}")
        print(f"      Preview: {corpus[doc_id][:80].replace('\n', ' ')}...\n")

# 2. Phân tích kĩ Failure Case đặc biệt: "heart attack" vs "myocardial infarction"
print("=" * 80)
print("CHI TIẾT FAILURE CASE (LEXICAL GAP / SYNONYM PROBLEM)")
print("=" * 80)

q_fail = "heart attack treatment"
q_fail_vec = vectorizer_search.transform([q_fail])

# Tìm thử các doc có chứa thuật ngữ y khoa "myocardial infarction"
synonym_doc_ids = [i for i, doc in enumerate(corpus) if "myocardial" in doc.lower() or "infarction" in doc.lower()]

if synonym_doc_ids:
    sample_syn_id = synonym_doc_ids[0]
    score_syn = cosine_similarity(q_fail_vec, X_search[sample_syn_id])[0][0]
    print(f"Document chứa 'myocardial infarction' (Doc ID: {sample_syn_id}):")
    print(f" - Nội dung: {corpus[sample_syn_id][:120]}...")
    print(f" - Score với query '{q_fail}': {score_syn:.4f}")
    print(" ➔ GIẢI THÍCH: Mặc dù đồng nghĩa, điểm Similarity tiệm cận/bằng 0 do Lexical Overlap = 0!")
else:
    print("Không tìm thấy document chứa 'myocardial infarction' trong 30K corpus.")

PART I: ERROR ANALYSIS - CHẠY THỬ VÀ PHÂN TÍCH 4 QUERIES

📌 TYPE: [Good (Hiệu quả)] | QUERY: 'medical image classification'
--------------------------------------------------------------------------------
Rank  | Doc ID   | Similarity | Matched Terms (Lexical Overlap)
--------------------------------------------------------------------------------
1     | 18971    | 0.4237     | classification(IDF:6.70)
      Preview: The new RTS Environmental Classification system (RTS GLT) is designed for partie...

2     | 8527     | 0.3537     | classification(IDF:6.70)
      Preview: History of maize classification. How races used in classification. Geographical ...

3     | 19908    | 0.2609     | image(IDF:4.33)
      Preview: Download League Of Legends Wallpapers in high-quality for your desktop and smart...

4     | 12658    | 0.2490     | medical(IDF:4.36)
      Preview: This guidance is for pharmacists who handle, use and sell/supply medical devices...

5     | 15682    | 0.2486     | image(